<a href="https://colab.research.google.com/github/NeoRedcraft/thesis-project-1/blob/main/ResNet50_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialize Model

In [2]:
# Github Repo Initialize
!git clone https://github.com/NeoRedcraft/thesis-project-1
# Mount Google drive
from google.colab import drive
drive.mount('/content/drive')

Cloning into 'thesis-project-1'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 15 (delta 2), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 4.23 KiB | 4.23 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
# Download requirements.txt
!pip install -r thesis-project-1/requirements.txt

In [2]:
# Import Libaries
import torch
import torchvision
# Import specific functions
from torchvision.models import resnet50 #ResNet50 model initialize
from torchvision.datasets import ImageFolder #Import Image Dataset
from torchvision.transforms import transforms # Image Transforming
from torch.utils.data import DataLoader #Image Dataset spiltting and shuffling
from torch.nn import nn #Model Math Calculation

| Parameter | Planned Value | Description |
|---|---|---|
| Input image size | 224 × 224 |Required by MobileNetV2 and ResNet-50 |
| Batch size | 32 | Number of images processed per training step |
| Number of epochs | 20-30 | Planned training iterations |
| Optimizer | Adam | Gradient optimizer |
| Learning rate | 0.001 | Initial learning rate |
| Loss function | Categorical Cross-Entropy | loss for multi-classification |
| Output classes | 3 | Staple, Snack, Beverage |
| Transfer learning | ImageNet pretrained weights | Used for feature initialization |

# 1. Configure the Transforming of the Dataset

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# 2. Load the Dataset

In [ ]:
train_dataset = ImageFolder(root="dataset/train/", transform=train_transforms)
val_dataset   = ImageFolder(root="dataset/valid/", transform=val_transforms)

print(train_dataset.classes)  # ['beverage', 'snack', 'staple']

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

# 3. Initialize Hyperparameters

In [ ]:
MAX_EPOCHS  = 30        # MIGHT NEED TO CHANGE, CUZ TOO SMALL FOR TRAINING
LR          = 0.001
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 4. Implement Early Stopping During Traning

- Not stated in the thesis paper
- Uniquely added

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, model_name="model"):
        """
        patience  : how many epochs to wait after last improvement
        min_delta : minimum change to qualify as an improvement
        """
        self.patience    = patience
        self.min_delta   = min_delta
        self.model_name  = model_name
        self.counter     = 0          # epochs without improvement
        self.best_loss   = None
        self.early_stop  = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            # First epoch — set baseline
            self.best_loss = val_loss
            self.save_checkpoint(model)

        elif val_loss < self.best_loss - self.min_delta:
            # Improvement found — reset counter
            print(f"   ✔ Val loss improved "
                  f"({self.best_loss:.4f} → {val_loss:.4f}). Saving model.")
            self.best_loss = val_loss
            self.counter   = 0
            self.save_checkpoint(model)

        else:
            # No improvement — increment counter
            self.counter += 1
            print(f"   ✘ No improvement. "
                  f"Patience: [{self.counter}/{self.patience}]")

            if self.counter >= self.patience:
                print(f"\n Early stopping triggered for {self.model_name} "
                      f"after {self.patience} epochs without improvement.")
                self.early_stop = True

    def save_checkpoint(self, model):
        # Saves the best model weights to disk
        torch.save(model.state_dict(),
                   f"best_{self.model_name}.pth")

# 5. Initialize Model (Resnet50)

In [ ]:
resnet = torchvision.models.resnet50(pretrained=True)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet = resnet.to(device)

# 6. Loss & Optimizers

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer_resnet = torch.optim.Adam(resnet.parameters(), lr=LR)

# 7. Training Function with Early Stopping

In [ ]:
def train_model(model, optimizer, model_name):
    print(f"\n{'='*50}")
    print(f" Training: {model_name}")
    print(f"{'='*50}")

    early_stopping = EarlyStopping(
        patience=5,       # stop after 5 epochs of no improvement
        min_delta=0.001,  # improvement must be at least 0.001 to count
        model_name=model_name
    )

    for epoch in range(MAX_EPOCHS):
        # ── Training Phase ──
        model.train()
        running_loss = 0.0
        correct = 0
        total   = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted  = torch.max(outputs, 1)
            correct       += (predicted == labels).sum().item()
            total         += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc  = 100 * correct / total

        # ── Validation Phase ──
        model.eval()
        val_loss    = 0.0
        val_correct = 0
        val_total   = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs        = model(images)
                loss           = criterion(outputs, labels)

                val_loss    += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total   += labels.size(0)

        val_loss = val_loss / len(val_loader)
        val_acc  = 100 * val_correct / val_total

        print(f"\nEpoch [{epoch+1:02d}/{MAX_EPOCHS}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # ── Check Early Stopping ──
        early_stopping(val_loss, model)

        if early_stopping.early_stop:
            print(f"\n>>> Stopped at epoch {epoch+1} out of {MAX_EPOCHS}")
            break

    # ── Restore Best Weights ──
    model.load_state_dict(torch.load(f"best_{model_name}.pth"))
    print(f"\n✔ Best weights restored for {model_name}.")

# 8. Train Both Models

In [ ]:
train_model(resnet,    optimizer_resnet,    "ResNet50")